# Домашнее преимущество в S95

**Ноутбук 04 — домашняя локация определяется по факту**

---

Повторяет ноутбук 03 целиком, с одним отличием: домашней считается
не та локация, которую участник указал в анкете, а та, где он бегал чаще
всего. При равенстве берётся локация, куда он пришёл раньше


## 1. Загрузка

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path.cwd() / "data"

activities = pd.read_csv(DATA / "activities.csv", parse_dates=["date"])
results    = pd.read_csv(DATA / "results.csv", dtype={"athlete_id": "Int64"})
athletes   = pd.read_csv(DATA / "athletes.csv").rename(columns={"event_id": "home_event_id"})
events     = pd.read_csv(DATA / "events.csv")

EXCLUDE = {7, 17, 32,   # Брест, Гомель, Гродно
           16, 18,      # Белград, Нови-Сад
           38,          # Алматы
           31}          # сводная запись S95 & Friends
activities = activities[~activities["event_id"].isin(EXCLUDE)]

runs = (results.dropna(subset=["athlete_id"])
        .merge(activities[["id", "date", "event_id"]],
               left_on="activity_id", right_on="id")
        .drop(columns="id"))
runs["athlete_id"] = runs["athlete_id"].astype(int)

print(f"{len(runs):,} забегов, {runs['athlete_id'].nunique():,} спортсменов, "
      f"{runs['event_id'].nunique()} локаций")

125,242 забегов, 16,413 спортсменов, 31 локаций


## 2. Домашняя локация по факту

Правило Grunseit et al.: наиболее посещаемая локация за период наблюдения,
при равенстве — посещённая раньше.

In [2]:
visits = (runs.groupby(["athlete_id", "event_id"])
          .agg(n=("date", "size"), first_visit=("date", "min"))
          .reset_index())

home = (visits.sort_values(["n", "first_visit"], ascending=[False, True])
        .drop_duplicates("athlete_id")
        .set_index("athlete_id")["event_id"])

print(f"дом определён для {len(home):,} человек, пропусков {home.isna().sum()}")

дом определён для 16,413 человек, пропусков 0


Сравним с анкетой: у тех, кто её заполнил, два определения должны
совпадать достаточно часто, иначе фактическое измеряет не принадлежность,
а случайную географию.

In [3]:
declared = athletes.set_index("id")["home_event_id"].dropna()
common = home.index.intersection(declared.index)

agree = (home.loc[common] == declared.loc[common]).mean()
print(f"сравнимо {len(common):,} человек, совпадает {agree:.1%}")

сравнимо 4,516 человек, совпадает 92.0%


## 3. Гостевые забеги и рабочая выборка

In [4]:
runs["home_event_id"] = runs["athlete_id"].map(home)
runs["away"] = runs["event_id"] != runs["home_event_id"]
runs["home"] = (~runs["away"]).astype(float)

per = runs.groupby("athlete_id")["away"].agg(runs="size", away="sum")
per["home_runs"] = per["runs"] - per["away"]

movers = per[(per["away"] > 0) & (per["home_runs"] > 0)].index
panel = runs[runs["athlete_id"].isin(movers)].copy()

print(f"бегали в гостях: {(per['away'] > 0).sum():,}")
print(f"из них ни разу дома: {((per['away'] > 0) & (per['home_runs'] == 0)).sum()}")
print(f"movers: {len(movers):,}")
print(f"панель: {len(panel):,} забегов, из них гостевых {int(panel['away'].sum()):,}")

бегали в гостях: 2,687
из них ни разу дома: 0
movers: 2,687
панель: 54,293 забегов, из них гостевых 11,738


Ни разу не бежавших дома здесь быть не может: наиболее посещаемая локация
по построению собирает хотя бы один забег. Это первое отличие от ноутбука 03,
где таких оказалось 190.

## 4. Эффекты

In [5]:
panel["month"] = panel["date"].dt.month

coords = events.set_index("id")[["latitude", "longitude"]]

def haversine(lat1, lon1, lat2, lon2):
    R, p = 6371.0, np.pi / 180
    a = (np.sin((lat2 - lat1) * p / 2) ** 2
         + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

panel["dist_km"] = haversine(
    panel["home_event_id"].map(coords["latitude"]),
    panel["home_event_id"].map(coords["longitude"]),
    panel["event_id"].map(coords["latitude"]),
    panel["event_id"].map(coords["longitude"]),
).round(1)

print(f"пропусков dist_km: {panel['dist_km'].isna().sum()}")
panel.loc[panel["away"], "dist_km"].describe().round(1)

пропусков dist_km: 665


count    11166.0
mean        42.9
std        281.8
min          1.1
25%          3.6
50%          6.0
75%         17.9
max       6252.0
Name: dist_km, dtype: float64

## 5. Прямое сравнение

In [6]:
panel.groupby("away")["total_time"].agg(["size", "mean", "median"]).round(1)

,size,mean,median
away,,,
False,42555,1698.9,1652.0
True,11738,1660.2,1626.0


In [7]:
w = panel.groupby(["athlete_id", "away"])["total_time"].mean().unstack()
(w[True] - w[False]).describe().round(1)

count    2687.0
mean      -20.4
std       236.8
min     -1602.5
25%      -134.8
50%       -25.5
75%        82.3
max      1485.7
dtype: float64

Та же проверка на тех, у кого не меньше трёх забегов с каждой стороны:

In [8]:
d = (w[True] - w[False]).rename("diff").to_frame().join(per)
d[(d["away"] >= 3) & (d["home_runs"] >= 3)]["diff"].describe().round(1)

count    917.0
mean     -23.9
std      157.4
min     -693.7
25%     -110.6
50%      -25.5
75%       54.2
max      704.8
Name: diff, dtype: float64

## 6. Спецификации



In [9]:
import pyfixest as pf

CL = {"CRV1": "event_id"}

m1 = pf.feols("total_time ~ home", data=panel, vcov=CL)
m2 = pf.feols("total_time ~ home | month", data=panel, vcov=CL)
m3 = pf.feols("total_time ~ home | athlete_id", data=panel, vcov=CL)
m4 = pf.feols("total_time ~ home | athlete_id + event_id", data=panel, vcov=CL)
m5 = pf.feols("total_time ~ home | athlete_id + activity_id", data=panel, vcov=CL)

pf.etable([m1, m2, m3, m4, m5])

/usr/local/lib/python3.13/dist-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 383 singleton fixed effect(s) dropped from the model.
  warnings.warn(


GT(_tbl_data=  __index_level_0__ __index_level_1__                          0  \
0              coef              home       38.703 <br> (29.281)   
1              coef         Intercept  1660.206*** <br> (22.738)   
2                fe       activity_id                          -   
3                fe        athlete_id                          -   
4                fe          event_id                          -   
5                fe             month                          -   
6             stats      Observations                     54,293   
7             stats                R²                      0.002   

                      1                     2                       3  \
0  35.387 <br> (27.437)  22.029 <br> (15.766)  35.171*** <br> (8.788)   
1                                                                       
2                     -                     -                       -   
3                     -                     x                       -   
4                     -                     -                       -   
5                     x                     -                       -   
6                54,293                54,293                  54,293   
7                 0.011                  0.61                   0.613   

                        4  
0  29.447*** <br> (6.507)  
1                          
2                       -  
3                       -  
4                       -  
5                       -  
6                  53,910  
7                   0.665  , _body=<great_tables._gt_data.Body object at 0x7f9046867620>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align=None, column_width=None, stub_level=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None, stub_level=0), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None, stub_level=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None, stub_level=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None, stub_level=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(4)'), column_align='center', column_width=None, stub_level=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(5)'), column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f90466dee90>, _spanners=Spanners([SpannerInfo(spanner_id='total_time', spanner_level=1, spanner_label=Html(text='total_time'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3', '4'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f90466681a0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f90466df250>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f90466682f0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7f90466686e0>, <great_tables._gt_data.FormatInfo object at 0x7f90466df890>, <great_tables._gt_data.FormatInfo object at 0x7f90466dfc50>, <great_tables._gt_data.FormatInfo object at 0x7f9046768510>, <great_tables._gt_data.FormatInfo object at 0x7f90467688a0>, <great_tables._gt_data.FormatInfo object at 0x7f9046783e30>, <great_tables._gt_data.FormatInfo object at 0x7f90467069c0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsIn

## 7. Сравнение двух определений дома

Главный вопрос ноутбука: меняется ли вывод, если дом брать по факту,
а не по анкете.

In [11]:
comparison = pd.DataFrame({
    "по анкете (ноутбук 03)": [5578, 4516, 1349, 190, 1159, 39883, 8900,
                               43.6, 30.0, 43.6, 37.0],
    "по факту (этот ноутбук)": [None] * 11,
}, index=["указали / определён дом", "из них с забегами", "бегали в гостях",
          "ни разу не бежали дома", "movers", "забегов в панели", "из них гостевых",
          "alpha (1) ничего", "alpha (3) спортсмен",
          "alpha (4) спортсмен+локация", "alpha (5) спортсмен+забег"])

comparison["по факту (этот ноутбук)"] = [
    len(home), len(per), int((per["away"] > 0).sum()),
    int(((per["away"] > 0) & (per["home_runs"] == 0)).sum()),
    len(movers), len(panel), int(panel["away"].sum()),
    round(m1.coef()["home"], 1), round(m3.coef()["home"], 1),
    round(m4.coef()["home"], 1), round(m5.coef()["home"], 1),
]
comparison

,по анкете (ноутбук 03),по факту (этот ноутбук)
указали / определён дом,5578.0,16413.0
из них с забегами,4516.0,16413.0
бегали в гостях,1349.0,2687.0
ни разу не бежали дома,190.0,0.0
movers,1159.0,2687.0
забегов в панели,39883.0,54293.0
из них гостевых,8900.0,11738.0
alpha (1) ничего,43.6,38.7
alpha (3) спортсмен,30.0,22.0
alpha (4) спортсмен+локация,43.6,35.2
